# Milestone 6: Shared memory for the denoise stage

## The problem

In the denoise kernel, each pixel reads its 3 × 3 neighbourhood (9 values) for each of 3 colours. But each of those values is *also* needed by the 8 surrounding pixels. So the same value is requested from global memory about **9 times**, once by each thread whose 3 × 3 window covers it.

For one block of 32 × 8 = 256 pixels:
- **global-memory version:** 256 pixels × 9 neighbours × 3 colours = **6,912 global reads**
- **shared-memory version:** the block's patch plus a 1-pixel border is 34 × 10 = 340 values × 3 colours = **1,020 global reads**, then all 6,912 neighbour reads come from shared memory

That's **6.8× fewer global memory reads.**

## Shared memory, the halo, and `__syncthreads`

**Shared memory** is a small, very fast memory inside each GPU core (SM), shared by the threads of one block. Think of it as a **workbench**: instead of each worker walking to the warehouse (global memory) nine times, the team fetches all the parts once, lays them on the bench, and then everyone works from the bench.

```
        34 columns in shared memory
     +----------------------------------+
     | h  h  h  h  ...  h  h  h  h  h   |   <- halo row (from the block above)
     | h  P  P  P  ...  P  P  P  P  h   |
     | h  P  P  P  ...  P  P  P  P  h   |   8 rows of P = the 32 x 8 pixels
     | ...                              |   this block computes
     | h  P  P  P  ...  P  P  P  P  h   |
     | h  h  h  h  ...  h  h  h  h  h   |   <- halo row (from the block below)
     +----------------------------------+
       ^ halo column                ^ halo column
```

The **halo** (`h`) is the 1-pixel border. The pixels on the edge of the block need neighbours that belong to the next block, so the tile must include them.

The kernel works in two phases:
1. **Load.** All 256 threads together copy the 340 values per colour into shared memory. There are more values than threads, so some threads load two.
2. **`__syncthreads()`.** Every thread waits until the whole block has finished loading. Without this, a fast thread could read a neighbour that hasn't been loaded yet and get garbage.
3. **Compute.** Each thread reads its 9 neighbours from shared memory.

**A trap the code avoids:** threads whose pixel lies outside the image must *not* `return` before `__syncthreads()`. They're still needed to load the halo, and if they left early, the other threads would wait at the barrier forever. So the bounds check comes after the barrier.

## Honest prediction

On the T4, shared memory and the **L1 cache** are built from the same on-chip memory. The Milestone 5 denoise was probably already getting many repeated neighbour reads from L1 rather than from main memory. Shared memory makes the reuse *guaranteed* and saves the 9 edge-clamp calculations per pixel, but the speed-up may be smaller than the 6.8× read reduction suggests. **The measurement decides.**

The program measures each version **twice, alternating** (E, F, E, F), so a slow drift in GPU clock speed can't unfairly favour one of them.

**Runtime must be T4 GPU.**

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project

## Step 2: Write the new files

In [ ]:
%%writefile gpu_shared_denoise.h
// gpu_shared_denoise.h
// Milestone 6: the denoise stage using shared memory (planar layout).

#pragma once

// The block size MUST match the tile size, because each thread computes
// exactly one output pixel of the tile. We use the best shape from Milestone 5.
const int TILE_WIDTH = 32;
const int TILE_HEIGHT = 8;

// The tile in shared memory is 1 pixel bigger on every side (the "halo"),
// because the pixels on the edge of the tile need neighbours from outside it.
const int HALO_SIZE = 1;
const int SHARED_TILE_WIDTH = TILE_WIDTH + 2 * HALO_SIZE;    // 34
const int SHARED_TILE_HEIGHT = TILE_HEIGHT + 2 * HALO_SIZE;  // 10

__global__ void denoiseSharedKernel(const float* rgbPlanes, float* denoisedPlanes,
                                    int imageWidth, int imageHeight);


In [ ]:
%%writefile gpu_shared_denoise.cu
// gpu_shared_denoise.cu
// Denoise with shared memory.
//
// The problem with the naive denoise: each pixel reads its 9 neighbours from
// global memory, and each of those values is ALSO read by the 8 surrounding
// threads. So every value is requested from global memory about 9 times.
//
// The fix: each block first copies its patch of the image (its 32x8 pixels plus
// a 1-pixel border, 34x10 values) into SHARED MEMORY, once. Then every thread
// reads its 9 neighbours from shared memory instead.
//
// Shared memory is a small, very fast memory INSIDE each GPU core (SM). It is
// shared by all threads of ONE block, and it only exists while that block runs.
// Global memory is large (15 GB on the T4) but far away and much slower.

#include "gpu_shared_denoise.h"

static __device__ int clampIndexShared(int index, int size) {
    if (index < 0) {
        return 0;
    }
    if (index >= size) {
        return size - 1;
    }
    return index;
}

__global__ void denoiseSharedKernel(const float* rgbPlanes, float* denoisedPlanes,
                                    int imageWidth, int imageHeight) {
    // "__shared__" puts this array in shared memory. There is ONE copy per
    // block, seen by all of the block's threads (unlike normal local variables,
    // where every thread has its own private copy).
    // 3 channels x 10 rows x 34 columns x 4 bytes = 4080 bytes per block.
    __shared__ float sharedTile[3][SHARED_TILE_HEIGHT][SHARED_TILE_WIDTH];

    int pixelCount = imageWidth * imageHeight;

    // Top-left image position of this block's 32x8 area.
    int blockStartColumn = blockIdx.x * TILE_WIDTH;
    int blockStartRow = blockIdx.y * TILE_HEIGHT;

    // ---- Phase 1: the whole block loads the tile together ----
    // The tile has 34 x 10 = 340 values per channel, but the block has only
    // 32 x 8 = 256 threads. So we number the threads 0..255 and let thread t
    // load values t, t + 256, t + 512, ... until all 340 are loaded.
    // Consecutive threads load consecutive columns, so the loads are coalesced.
    int threadNumber = threadIdx.y * TILE_WIDTH + threadIdx.x;
    int threadsPerBlock = TILE_WIDTH * TILE_HEIGHT;
    int valuesPerTile = SHARED_TILE_WIDTH * SHARED_TILE_HEIGHT;

    for (int loadNumber = threadNumber; loadNumber < valuesPerTile; loadNumber = loadNumber + threadsPerBlock) {
        int tileRow = loadNumber / SHARED_TILE_WIDTH;
        int tileColumn = loadNumber % SHARED_TILE_WIDTH;

        // Tile position 0 is one pixel ABOVE / LEFT of the block (the halo),
        // hence "- HALO_SIZE". Clamp so halo pixels outside the image use the
        // nearest edge pixel, exactly like the CPU version.
        int imageRow = clampIndexShared(blockStartRow + tileRow - HALO_SIZE, imageHeight);
        int imageColumn = clampIndexShared(blockStartColumn + tileColumn - HALO_SIZE, imageWidth);
        int imagePixel = imageRow * imageWidth + imageColumn;

        for (int channel = 0; channel < 3; channel++) {
            sharedTile[channel][tileRow][tileColumn] = rgbPlanes[channel * pixelCount + imagePixel];
        }
    }

    // "__syncthreads()" is a barrier: every thread of the block waits here
    // until ALL threads of the block have reached it. Without it, a fast thread
    // could start reading neighbours that a slower thread has not loaded yet.
    //
    // IMPORTANT: every thread must reach __syncthreads, including threads
    // whose pixel is outside the image (they still helped load the tile). That
    // is why the "outside the image? return" check comes AFTER this line.
    // If some threads returned early, the others would wait here forever.
    __syncthreads();

    // ---- Phase 2: each thread computes its own pixel from shared memory ----
    int column = blockStartColumn + threadIdx.x;
    int row = blockStartRow + threadIdx.y;
    if (column >= imageWidth || row >= imageHeight) {
        return;
    }

    const float gaussianWeights[3][3] = {
        {1.0f, 2.0f, 1.0f},
        {2.0f, 4.0f, 2.0f},
        {1.0f, 2.0f, 1.0f}
    };
    const float weightSum = 16.0f;

    // This thread's pixel sits at (threadIdx.y + 1, threadIdx.x + 1) in the
    // tile, because of the 1-pixel halo above and to the left.
    int centreTileRow = threadIdx.y + HALO_SIZE;
    int centreTileColumn = threadIdx.x + HALO_SIZE;
    int pixelIndex = row * imageWidth + column;

    for (int channel = 0; channel < 3; channel++) {
        float weightedTotal = 0.0f;

        // Same loop order as the naive kernel, so the sum is added up in the
        // same order and the result should be bit-for-bit identical.
        for (int rowOffset = -1; rowOffset <= 1; rowOffset++) {
            for (int columnOffset = -1; columnOffset <= 1; columnOffset++) {
                float neighbourValue = sharedTile[channel][centreTileRow + rowOffset][centreTileColumn + columnOffset];
                float weight = gaussianWeights[rowOffset + 1][columnOffset + 1];
                weightedTotal = weightedTotal + neighbourValue * weight;
            }
        }

        denoisedPlanes[channel * pixelCount + pixelIndex] = weightedTotal / weightSum;
    }
}


In [ ]:
%%writefile gpu_shared_experiment_main.cu
// gpu_shared_experiment_main.cu
// Milestone 6: compares the planar 32x8 pipeline (best from Milestone 5) with
// the same pipeline using the shared-memory denoise kernel.
// It also checks that both denoise kernels give exactly the same floats.

#include <cmath>
#include <iomanip>
#include <iostream>
#include <vector>

#include "cuda_helpers.h"
#include "gpu_buffers.h"
#include "gpu_kernels.h"
#include "gpu_planar_kernels.h"
#include "gpu_shared_denoise.h"
#include "isp_io.h"

const int STAGE_COUNT = 5;
const int TIMED_RUN_COUNT = 10;

static void runPipelineOnce(DeviceBuffers& buffers, float* deviceDenoisedTarget,
                            const ImageMetadata& metadata, bool useSharedDenoise,
                            dim3 gridShape, dim3 blockShape,
                            cudaEvent_t stageEvents[], float stageMilliseconds[]) {
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;

    checkCudaError(cudaEventRecord(stageEvents[0]), "recording event 0");

    blackLevelKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceNormalised,
                                                imageWidth, imageHeight,
                                                metadata.blackLevel, metadata.whiteLevel);
    checkCudaError(cudaGetLastError(), "launching blackLevelKernel");
    checkCudaError(cudaEventRecord(stageEvents[1]), "recording event 1");

    demosaicPlanarKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                                    imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching demosaicPlanarKernel");
    checkCudaError(cudaEventRecord(stageEvents[2]), "recording event 2");

    whiteBalancePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                        metadata.whiteBalanceRed,
                                                        metadata.whiteBalanceGreen,
                                                        metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching whiteBalancePlanarKernel");
    checkCudaError(cudaEventRecord(stageEvents[3]), "recording event 3");

    if (useSharedDenoise) {
        denoiseSharedKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, deviceDenoisedTarget,
                                                       imageWidth, imageHeight);
    } else {
        denoisePlanarKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, deviceDenoisedTarget,
                                                       imageWidth, imageHeight);
    }
    checkCudaError(cudaGetLastError(), "launching denoise");
    checkCudaError(cudaEventRecord(stageEvents[4]), "recording event 4");

    gammaPlanarKernel<<<gridShape, blockShape>>>(deviceDenoisedTarget, buffers.deviceOutput,
                                                 imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching gammaPlanarKernel");
    checkCudaError(cudaEventRecord(stageEvents[5]), "recording event 5");

    checkCudaError(cudaEventSynchronize(stageEvents[STAGE_COUNT]), "waiting for the pipeline");
    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        checkCudaError(cudaEventElapsedTime(&stageMilliseconds[stage],
                                            stageEvents[stage], stageEvents[stage + 1]),
                       "reading stage time");
    }
}

// 1 warm-up run, then TIMED_RUN_COUNT runs, averaged per stage, printed as one row.
static void measureAndPrint(const char* rowName, DeviceBuffers& buffers, float* deviceDenoisedTarget,
                            const ImageMetadata& metadata, bool useSharedDenoise,
                            dim3 gridShape, dim3 blockShape, cudaEvent_t stageEvents[]) {
    float thisRunMilliseconds[STAGE_COUNT];
    float averageMilliseconds[STAGE_COUNT] = {0.0f, 0.0f, 0.0f, 0.0f, 0.0f};

    runPipelineOnce(buffers, deviceDenoisedTarget, metadata, useSharedDenoise,
                    gridShape, blockShape, stageEvents, thisRunMilliseconds);

    for (int runNumber = 0; runNumber < TIMED_RUN_COUNT; runNumber++) {
        runPipelineOnce(buffers, deviceDenoisedTarget, metadata, useSharedDenoise,
                        gridShape, blockShape, stageEvents, thisRunMilliseconds);
        for (int stage = 0; stage < STAGE_COUNT; stage++) {
            averageMilliseconds[stage] = averageMilliseconds[stage] + thisRunMilliseconds[stage];
        }
    }

    float totalMilliseconds = 0.0f;
    std::cout << std::setw(24) << rowName;
    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        averageMilliseconds[stage] = averageMilliseconds[stage] / TIMED_RUN_COUNT;
        totalMilliseconds = totalMilliseconds + averageMilliseconds[stage];
        std::cout << std::setw(10) << averageMilliseconds[stage];
    }
    std::cout << std::setw(10) << totalMilliseconds << std::endl;
}

int main() {
    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }

    DeviceBuffers buffers;
    allocateDeviceBuffers(buffers, pixelCount);
    checkCudaError(cudaMemcpy(buffers.deviceBayer, bayerPixels.data(), buffers.bayerBytes,
                              cudaMemcpyHostToDevice), "copying Bayer data to the GPU");

    // A second denoise output buffer, so we can keep both results and compare them.
    float* deviceDenoisedShared = nullptr;
    checkCudaError(cudaMalloc(&deviceDenoisedShared, buffers.rgbFloatBytes),
                   "allocating second denoise buffer");

    cudaEvent_t stageEvents[STAGE_COUNT + 1];
    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventCreate(&stageEvents[eventNumber]), "creating stage event");
    }

    dim3 blockShape(TILE_WIDTH, TILE_HEIGHT);
    int blocksAcross = (metadata.imageWidth + TILE_WIDTH - 1) / TILE_WIDTH;
    int blocksDown = (metadata.imageHeight + TILE_HEIGHT - 1) / TILE_HEIGHT;
    dim3 gridShape(blocksAcross, blocksDown);

    std::cout << std::fixed << std::setprecision(3);
    std::cout << "Average kernel times in ms (" << TIMED_RUN_COUNT << " warm runs each)" << std::endl;
    std::cout << std::setw(24) << "version"
              << std::setw(10) << "black" << std::setw(10) << "demosaic"
              << std::setw(10) << "white b." << std::setw(10) << "denoise"
              << std::setw(10) << "gamma" << std::setw(10) << "TOTAL" << std::endl;

    // Measure each version twice, alternating, so a slow drift of the GPU clock
    // during the program cannot unfairly favour whichever version runs first.
    measureAndPrint("E planar 32x8 (global)", buffers, buffers.deviceDenoised, metadata, false,
                    gridShape, blockShape, stageEvents);
    measureAndPrint("F shared-memory denoise", buffers, deviceDenoisedShared, metadata, true,
                    gridShape, blockShape, stageEvents);
    measureAndPrint("E again", buffers, buffers.deviceDenoised, metadata, false,
                    gridShape, blockShape, stageEvents);
    measureAndPrint("F again", buffers, deviceDenoisedShared, metadata, true,
                    gridShape, blockShape, stageEvents);

    // ---- Compare the two denoise results float by float ----
    std::vector<float> denoisedGlobal(pixelCount * 3);
    std::vector<float> denoisedShared(pixelCount * 3);
    checkCudaError(cudaMemcpy(denoisedGlobal.data(), buffers.deviceDenoised, buffers.rgbFloatBytes,
                              cudaMemcpyDeviceToHost), "copying global-memory denoise result");
    checkCudaError(cudaMemcpy(denoisedShared.data(), deviceDenoisedShared, buffers.rgbFloatBytes,
                              cudaMemcpyDeviceToHost), "copying shared-memory denoise result");

    float largestFloatDifference = 0.0f;
    for (int valueIndex = 0; valueIndex < pixelCount * 3; valueIndex++) {
        float difference = std::fabs(denoisedGlobal[valueIndex] - denoisedShared[valueIndex]);
        if (difference > largestFloatDifference) {
            largestFloatDifference = difference;
        }
    }
    std::cout << std::setprecision(9);
    std::cout << "Largest float difference, global vs shared denoise: "
              << largestFloatDifference << std::endl;

    // The last run was F, so deviceOutput holds the shared-memory pipeline result.
    std::vector<unsigned char> outputBytes(buffers.outputBytes);
    checkCudaError(cudaMemcpy(outputBytes.data(), buffers.deviceOutput, buffers.outputBytes,
                              cudaMemcpyDeviceToHost), "copying output to the CPU");

    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventDestroy(stageEvents[eventNumber]), "destroying stage event");
    }
    checkCudaError(cudaFree(deviceDenoisedShared), "freeing second denoise buffer");
    freeDeviceBuffers(buffers);

    bool writeOk = writeRgbFile("output/gpu_shared_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    std::cout << "Saved output/gpu_shared_output.bin" << std::endl;
    return 0;
}


## Step 3: Compile
**Expected:** no output.

In [ ]:
!nvcc -O2 -arch=sm_75 -o /content/gpu_shared gpu_shared_experiment_main.cu gpu_shared_denoise.cu gpu_planar_kernels.cu gpu_buffers.cu gpu_pointwise_kernels.cu gpu_neighbourhood_kernels.cu cuda_helpers.cu isp_io.cpp
!g++ -O2 -std=c++17 -o /content/compare_outputs compare_outputs.cpp isp_io.cpp

## Step 4: Run (CHECKPOINT 1)
**Expected:** a table with 4 rows (E, F, E again, F again), then `Largest float difference, global vs shared denoise: 0.000000000` (both kernels add the same numbers in the same order, so they should match exactly), then `Saved output/gpu_shared_output.bin`.

**Paste the whole output to me.**

In [ ]:
!/content/gpu_shared

## Step 5: Check against the CPU (CHECKPOINT 2)
**Expected:** `RESULT: PASS`.

In [ ]:
!/content/compare_outputs output/cpu_output.bin output/gpu_shared_output.bin